# 1. Veri Seti Seçimi ve Tanıtımı (NASA C-MAPSS — FD001)

Bu notebook'ta NASA C-MAPSS (`FD001`) turbofan motor bozulma veri setinin eğitim (`train_FD001`), test (`test_FD001`) ve test kalan kullanım ömrü (`RUL_FD001`) dosyalarının temel yapısı, veri kalitesi, sabit/düşük varyanslı kanalları ve hedef değişken (RUL) ile sensör ilişkileri incelenmektedir. Veri sızıntısını önlemek amacıyla özellik ve davranış analizleri yalnızca eğitim seti üzerinden yürütülmüştür.

## 1.1 Veri Yükleme ve Temel Boyut Özeti
Eğitim (`train_FD001`), test (`test_FD001`) ve gerçek kalan ömür (`RUL_FD001`) dosyaları taşınabilir göreceli yollarla okunarak satır/sütun boyutları ve motor (`unit`) sayıları kontrol edilmektedir.

In [1]:
from pathlib import Path
import pandas as pd

index_cols = ["unit", "cycle"]
op_cols = ["op_set1", "op_set2", "op_set3"]
sensor_cols = [
    "T2",
    "T24",
    "T30",
    "T50",
    "P2",
    "P15",
    "P30",
    "Nf",
    "Nc",
    "epr",
    "Ps30",
    "phi",
    "NRf",
    "NRc",
    "BPR",
    "farB",
    "htBleed",
    "Nf_dmd",
    "PCNfR_dmd",
    "W31",
    "W32",
]
all_cols = index_cols + op_cols + sensor_cols

# Notebook'un konumuna göre data/ veya ../data/ klasörünü bul
data_dir = next(
    (
        p
        for p in [Path("data"), Path("../data")]
        if (p / "train_FD001.txt").exists()
    ),
    None,
)
if data_dir is None:
  raise FileNotFoundError(
      f"train_FD001.txt bulunamadı. Mevcut dizin: {Path.cwd()}"
  )

train_df = pd.read_csv(
    data_dir / "train_FD001.txt", sep=r"\s+", header=None, names=all_cols
)
test_df = pd.read_csv(
    data_dir / "test_FD001.txt", sep=r"\s+", header=None, names=all_cols
)
rul_df = pd.read_csv(
    data_dir / "RUL_FD001.txt", sep=r"\s+", header=None, names=["RUL"]
)

print("=== VERİ SETİ BOYUT VE YAPI ÖZETİ ===")
print(
    f"train_df -> şekil: {train_df.shape}, motor sayısı:"
    f" {train_df['unit'].nunique()}"
)
print(
    f"test_df  -> şekil: {test_df.shape}, motor sayısı:"
    f" {test_df['unit'].nunique()}"
)
print(f"rul_df   -> şekil: {rul_df.shape}")

=== VERİ SETİ BOYUT VE YAPI ÖZETİ ===
train_df -> şekil: (20631, 26), motor sayısı: 100
test_df  -> şekil: (13096, 26), motor sayısı: 100
rul_df   -> şekil: (100, 1)


## 1.2 Hedef ve Döngü Dağılımları
Eğitim setindeki motorların arızaya ulaşana kadar çalıştığı maksimum döngü (`max_cycle`) değeri ile dağılımlarının temel özet istatistikleri incelenmektedir.

In [2]:
max_cycles = train_df.groupby("unit")["cycle"].max()

print("=== DAĞILIM BİLGİLERİ ===")
print(
    f"Train max_cycle -> Min: {max_cycles.min()}, Max: {max_cycles.max()},"
    f" Ortalama: {max_cycles.mean():.2f}, Medyan: {max_cycles.median():.2f}"
)

=== DAĞILIM BİLGİLERİ ===
Train max_cycle -> Min: 128, Max: 362, Ortalama: 206.31, Medyan: 199.00


## 1.3 Veri Kalitesi ve Sabit / Düşük Varyanslı Kanallar
Eğitim ve test setlerinde eksik (`NaN`) ve mükerrer (tekrarlanmış) kayıtlar denetlenmekte; ardından yalnızca eğitim seti üzerinde tamamen sabit (`std == 0`) (standart sapma) veya çok düşük varyanslı kanallar tespit edilmektedir.

In [3]:
# Veri kalitesi kontrolleri (Train & Test temel sağlık kontrolü)
print("=== VERİ KALİTESİ KONTROLLERİ ===")
print("Train eksik (NaN) hücre sayısı        :", int(train_df.isna().sum().sum()))
print(
    "Train tam satır mükerrer kayıt sayısı :", int(train_df.duplicated().sum())
)
print(
    "Train aynı (unit, cycle) tekrar sayısı:",
    int(train_df.duplicated(subset=["unit", "cycle"]).sum()),
)
print("Test eksik (NaN) hücre sayısı         :", int(test_df.isna().sum().sum()))
print(
    "Test tam satır mükerrer kayıt sayısı  :", int(test_df.duplicated().sum())
)
print(
    "Test aynı (unit, cycle) tekrar sayısı :",
    int(test_df.duplicated(subset=["unit", "cycle"]).sum()),
)

# Sadece eğitim seti üzerinde sabit / düşük varyanslı kanallar
feature_cols = op_cols + sensor_cols
stats = pd.DataFrame({
    "min": train_df[feature_cols].min(),
    "max": train_df[feature_cols].max(),
    "range": train_df[feature_cols].max() - train_df[feature_cols].min(),
    "std": train_df[feature_cols].std(),
    "nunique": train_df[feature_cols].nunique(),
})

print("\nTamamen sabit sütunlar (std == 0) [Train]:")
print(stats[stats["std"] == 0])

print("\nÇok düşük varyanslı sütunlar (0 < std < 0.01) [Train]:")
print(stats[(stats["std"] > 0) & (stats["std"] < 0.01)])

=== VERİ KALİTESİ KONTROLLERİ ===
Train eksik (NaN) hücre sayısı        : 0
Train tam satır mükerrer kayıt sayısı : 0
Train aynı (unit, cycle) tekrar sayısı: 0
Test eksik (NaN) hücre sayısı         : 0
Test tam satır mükerrer kayıt sayısı  : 0
Test aynı (unit, cycle) tekrar sayısı : 0

Tamamen sabit sütunlar (std == 0) [Train]:
               min      max  range  std  nunique
op_set3     100.00   100.00    0.0  0.0        1
T2          518.67   518.67    0.0  0.0        1
epr           1.30     1.30    0.0  0.0        1
Nf_dmd     2388.00  2388.00    0.0  0.0        1
PCNfR_dmd   100.00   100.00    0.0  0.0        1

Çok düşük varyanslı sütunlar (0 < std < 0.01) [Train]:
             min      max   range           std  nunique
op_set1  -0.0087   0.0087  0.0174  2.187313e-03      158
op_set2  -0.0006   0.0006  0.0012  2.930621e-04       13
P2       14.6200  14.6200  0.0000  5.329200e-15        1
P15      21.6000  21.6100  0.0100  1.388985e-03        2
farB      0.0300   0.0300  0.0000  

## 1.4 Aktif Sensör İstatistikleri ve RUL Korelasyonu
Sabit/neredeyse sabit kanallar dışındaki 14 aktif sensörün temel istatistikleri ile eğitim setinde türetilen lineer (`RUL_linear`) ve `125` döngü üst sınırlı (`RUL_capped125`) hedeflerle korelasyonları incelenmektedir.

In [4]:
active_sensors = [
    "T24",
    "T30",
    "T50",
    "P30",
    "Nf",
    "Nc",
    "Ps30",
    "phi",
    "NRf",
    "NRc",
    "BPR",
    "htBleed",
    "W31",
    "W32",
]

# Eğitim seti için her motorun maksimum döngüsünü bul ve RUL hesapla
train_eda = train_df.copy()
max_cycle_map = train_eda.groupby("unit")["cycle"].transform("max")
train_eda["RUL_linear"] = max_cycle_map - train_eda["cycle"]
train_eda["RUL_capped125"] = train_eda["RUL_linear"].clip(upper=125)

active_stats = pd.DataFrame({
    "min": train_eda[active_sensors].min(),
    "max": train_eda[active_sensors].max(),
    "std": train_eda[active_sensors].std(),
    "corr_RUL_linear": train_eda[active_sensors].corrwith(
        train_eda["RUL_linear"]
    ),
    "corr_RUL_capped125": train_eda[active_sensors].corrwith(
        train_eda["RUL_capped125"]
    ),
})

active_stats["abs_corr_linear"] = active_stats["corr_RUL_linear"].abs()
active_stats = active_stats.sort_values("abs_corr_linear", ascending=False)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
print("=== KALAN 14 SENSÖR İSTATİSTİK VE KORELASYON TABLOSU [Train] ===")
print(active_stats.drop(columns=["abs_corr_linear"]).round(4))

=== KALAN 14 SENSÖR İSTATİSTİK VE KORELASYON TABLOSU [Train] ===
               min        max      std  corr_RUL_linear  corr_RUL_capped125
Ps30       46.8500    48.5300   0.2671          -0.6962             -0.7752
T50      1382.2500  1441.4900   9.0006          -0.6789             -0.7572
phi       518.6900   523.3800   0.7376           0.6720              0.7489
P30       549.8500   556.0600   0.8851           0.6572              0.7330
BPR         8.3249     8.5848   0.0375          -0.6427             -0.7209
W32        22.8942    23.6184   0.1083           0.6357              0.7073
W31        38.1400    39.4300   0.1807           0.6294              0.7046
T24       641.2100   644.5300   0.5001          -0.6065             -0.6785
htBleed   388.0000   400.0000   1.5488          -0.6062             -0.6808
T30      1571.0400  1616.9100   6.1311          -0.5845             -0.6550
Nf       2387.9000  2388.5600   0.0710          -0.5640             -0.6246
NRf      2387.8800  238